# VRP Solver Benchmark — Colab Edition

Compares **MILP (CP-SAT)**, **OR-Tools GLS**, **cuOpt (GPU)**, and **Greedy** on:
- Synthetic CVRP (random instances)
- Solomon VRPTW (standard benchmark with published best-known solutions)

**cuOpt is optional.** OR-Tools + Greedy + MILP run entirely on CPU with no extra setup.
To include cuOpt, see the [cuOpt Setup](#cuopt-setup) cell below — it uses the NVIDIA NIM cloud API (free trial credits available).

---
**Runtime:** ~8 min without cuOpt · ~15 min with cuOpt (T4 GPU recommended for cuOpt)

## 1. Install dependencies

In [ ]:
# Install OR-Tools and plotting libraries
!pip install -q ortools numpy matplotlib pandas

# Clone the repo
import os
if not os.path.exists('optimize_routing'):
    !git clone -q https://github.com/bchadburn/optimize_routing.git
%cd optimize_routing

## 2. cuOpt Setup (optional)

Skip this cell if you only want OR-Tools + Greedy + MILP.

**To enable cuOpt via the NVIDIA NIM cloud API:**
1. Go to [build.nvidia.com/nvidia/cuopt](https://build.nvidia.com/nvidia/cuopt)
2. Sign in with a free NVIDIA account → click **Get API Key**
3. Paste the key below

The NIM API provides free trial credits — no GPU or Docker required on your end.

In [ ]:
import os

# Paste your NVIDIA API key here (or leave empty to skip cuOpt)
NVIDIA_API_KEY = ""  # e.g. "nvapi-xxxxxxxxxxxx"

USE_CUOPT = bool(NVIDIA_API_KEY)
if USE_CUOPT:
    os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
    # Install the cuOpt REST client
    !pip install -q --extra-index-url https://pypi.nvidia.com cuopt-sh-client
    print("cuOpt enabled (NIM cloud API)")
else:
    print("cuOpt skipped — running OR-Tools + Greedy + MILP only")

## 3. Synthetic CVRP Benchmark

In [ ]:
import sys
sys.path.insert(0, '.')  # make vrp_benchmark importable

from vrp_benchmark.benchmark import run as run_cvrp

# Sizes to benchmark. Reduce counts or n_eval for a faster run.
run_cvrp(
    customer_counts=[10, 20, 30, 50, 100],
    include_milp=True,
    include_cuopt=USE_CUOPT,
    nim_mode=True,          # NIM API mode (ignored if USE_CUOPT=False)
    n_eval=5,               # instances per cell (use 10 for full benchmark)
    ortools_time_s=5,
    milp_time_s=60,         # 60s per instance; budget stops MILP when 1hr total exceeded
    milp_budget_s=600,      # 10min MILP budget (reduce to 120 for quick runs)
    cuopt_time_s=10,
)

### CVRP Results — Solution Quality

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('results/vrp_benchmark.csv')

# Build a cost table: pivot to (n_customers × solver)
cost_pivot = df.pivot_table(index='n_customers', columns='solver', values='mean_cost')
print(cost_pivot.round(0).to_string())

# Plot: cost vs n_customers
fig, ax = plt.subplots(figsize=(10, 5))
colors = {'greedy': '#9467bd', 'ortools': '#2ca02c', 'cuopt': '#ff7f0e',
          'milp_exact': '#1f77b4', 'milp_feasible': '#aec7e8', 'dqn': '#d62728'}
for solver in cost_pivot.columns:
    vals = cost_pivot[solver].dropna()
    ax.plot(vals.index, vals.values, marker='o', label=solver,
            color=colors.get(solver, 'gray'))

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of customers (n)')
ax.set_ylabel('Mean route distance (km)')
ax.set_title('CVRP: Solution cost vs problem size')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/colab_cvrp_cost.png', dpi=120)
plt.show()

## 4. Solomon VRPTW Benchmark

One instance per family (C1, C2, R1, R2, RC1, RC2). Instances auto-download on first run (~1 MB total).

In [ ]:
from vrp_benchmark.benchmark_solomon import run as run_solomon

run_solomon(
    instance_names=['C101', 'C201', 'R101', 'R201', 'RC101', 'RC201'],
    include_cuopt=USE_CUOPT,
    nim_mode=True,
    ortools_time_s=30,
    cuopt_time_s=30,
)

### Solomon Results — Gap vs Best-Known Solution

In [ ]:
df_s = pd.read_csv('results/solomon_benchmark.csv')
df_s = df_s[df_s['feasible'] == True].copy()

instances = df_s['instance'].unique()
solvers = df_s['solver'].unique()
colors_s = {'greedy': '#9467bd', 'ortools': '#2ca02c', 'cuopt': '#ff7f0e'}

x = np.arange(len(instances))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
for i, solver in enumerate(solvers):
    vals = []
    for inst in instances:
        row = df_s[(df_s['instance'] == inst) & (df_s['solver'] == solver)]
        vals.append(row['gap_vs_bks_pct'].values[0] if len(row) else float('nan'))
    bars = ax.bar(x + i * width, vals, width, label=solver,
                  color=colors_s.get(solver, 'gray'), edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, max(val, 0) + 0.5,
                    f'{val:.1f}%', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + width)
ax.set_xticklabels(instances, fontsize=9)
ax.set_ylabel('Gap vs best-known solution (%)')
ax.set_title('Solomon VRPTW: Gap vs BKS (30s time limit each)')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/colab_solomon_gap.png', dpi=120)
plt.show()

## 5. Summary Tables

In [ ]:
print("=== CVRP: Mean cost by solver and problem size ===")
print(cost_pivot.round(0).to_string())

print("\n=== Solomon VRPTW: Gap vs BKS ===")
gap_pivot = df_s.pivot_table(index='instance', columns='solver', values='gap_vs_bks_pct')
print(gap_pivot.round(1).to_string())

print("\n=== Solomon VRPTW: Vehicles used vs BKS ===")
bks_veh = df_s[['instance','bks_vehicles']].drop_duplicates().set_index('instance')
veh_pivot = df_s.pivot_table(index='instance', columns='solver', values='n_vehicles_used')
print(pd.concat([bks_veh, veh_pivot], axis=1).to_string())